# M61 — MonoDGP teacher → MobileNetV4 A2 student

**Revision: M61-2026-09-22-r1. Run sections 1–12 in order on a CUDA GPU. No phone needed.**

Teacher: frozen M54 MonoDGP ResNet50 epoch100. Student: frozen A2 MonoDETR MobileNetV4 Medium epoch130. This is **two matched 10-epoch runs**, not a new 195-epoch training run. Native Car/Vehicle index is **1**, Pedestrian is **0**.

We cache train-only, unaugmented teacher/student outputs in separate processes. Both training arms use that exact unaugmented view; all GT losses still supervise both classes. Teacher geometry is attached only to matched Vehicle targets and only to components passing the train audit. No teacher logits, Pedestrian KD, temperature sweep, or student architecture change.

**Restart:** rerun from section 1. Verified caches are reused; training resumes from the last complete epoch. An interrupted epoch restarts. Keep the same revision/environment, paths, and output directory. Provenance conflicts stop rather than mixing runs. Detailed logs and checkpoints are on Drive. This notebook does not reset or delete prior repositories.


## 1. Mount Drive, paths, and durable command runner

Edit only dataset source candidates or selection-file paths if your Drive layout differs. Selection JSONs must point to the exact existing checkpoints. The notebook derives model configs from pinned upstream source; no old runtime YAML is needed.


In [ ]:
# M61-2026-09-22-r1 — latest workflow marker; must print when this cell runs.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import hashlib, json, os, shlex, shutil, subprocess, sys, zipfile

REVISION = 'M61-2026-09-22-r1'
MOBILE_REPO = Path('/content/mobile_adas3d')
STUDENT_REPO = Path('/content/MonoDETR_M61')
TEACHER_REPO = Path('/content/MonoDGP_M61')
DRIVE = Path('/content/drive/MyDrive')
SPLIT_DIR = DRIVE / 'mobile_adas3d_splits/kitti_chen'
A2_SELECTION = DRIVE / 'mobile_adas3d_outputs/students/monodetr_a2_gt/product_checkpoint_sweep/a2_product_selection.json'
M54_SELECTION = DRIVE / 'mobile_adas3d_outputs/challengers/monodgp_m54/product_checkpoint_sweep/m54_product_selection.json'
DATASET_ROOT = Path('/content/kitti_m61')
DATASET_CANDIDATES = [
    Path('/content/kitti'), Path('/content/monodetr_kitti'), Path('/content/kitti_h1'),
    DRIVE / 'datasets/kitti',
]
OUTPUT_ROOT = DRIVE / 'mobile_adas3d_outputs/students/monodgp_to_monodetr_m61'
MANIFEST = OUTPUT_ROOT / 'm61_manifest.json'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run_logged(command, name, cwd=None):
    command = [str(x) for x in command]
    log_path = OUTPUT_ROOT / 'colab_logs' / (name + '.log')
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('+', shlex.join(command), '\nDurable log:', log_path, flush=True)
    tail = deque(maxlen=80)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    with log_path.open('a', buffering=1) as log:
        log.write('\nSTART ' + datetime.now(timezone.utc).isoformat() + '\n')
        p = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
            tail.append(line.rstrip())
        result = p.wait()
    if result:
        raise RuntimeError(f'Exit {result}; log={log_path}\n' + '\n'.join(tail))
    return log_path

def pilot(script, *args):
    return run_logged([sys.executable, '-u', MOBILE_REPO / 'scripts' / script,
                       '--manifest', MANIFEST, *args], Path(script).stem + '_' + '_'.join(str(x) for x in args).replace('/', '_'),
                      cwd=MOBILE_REPO)

print(REVISION, '\nTeacher:', TEACHER_REPO, '\nStudent:', STUDENT_REPO, '\nOutput:', OUTPUT_ROOT)
run_logged(['nvidia-smi'], 'gpu')


## 2. Update main, install dependencies, and prepare isolated source checkouts

Uses the correct repositories: **PuFanqi23/MonoDGP** for the teacher and **ZrrSkywalker/MonoDETR** for the student. CUDA extensions are built inside each repository, avoiding a shared global extension install. First run can take several minutes.


In [ ]:
url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE_REPO.exists():
    run_logged(['git', 'clone', '--branch', 'main', url, MOBILE_REPO], 'clone_mobile')
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=MOBILE_REPO, text=True).strip()
if branch != 'main':
    raise RuntimeError(f'Mobile repo is on {branch!r}; use a clean main checkout, not an old experiment branch.')
run_logged(['git', 'pull', '--ff-only'], 'update_mobile', cwd=MOBILE_REPO)
run_logged([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy',
            'opencv-python-headless', 'numba', 'scikit-image', 'scikit-learn',
            'tqdm', 'ninja', 'timm==1.0.20', 'pandas', 'tensorboard'],
           'dependencies')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
run_logged([sys.executable, '-u', MOBILE_REPO / 'scripts/setup_m61_sources.py',
            '--student-repo', STUDENT_REPO, '--teacher-repo', TEACHER_REPO],
           'source_and_cuda_setup', cwd=MOBILE_REPO)


## 3. Restore the KITTI view and check required saved selections

Images, original labels, and calibrations are reused; no teacher prediction is used as ground truth. Local KITTI files are preferred for speed, with the old Drive location as fallback.


In [ ]:
sources = {}
for name in ('image_2', 'label_2', 'calib'):
    alternatives = [f'training/{name}']
    if name in ('image_2', 'label_2'):
        alternatives.append(f'training/{name[:-1]}02')
    candidates = [root / relative for root in DATASET_CANDIDATES for relative in alternatives]
    sources[name] = next((p for p in candidates if p.is_dir()), None)
if any(p is None for p in sources.values()):
    raise FileNotFoundError(f'Missing KITTI directories: {sources}. Edit DATASET_CANDIDATES in section 1.')
(DATASET_ROOT / 'training').mkdir(parents=True, exist_ok=True)
(DATASET_ROOT / 'ImageSets').mkdir(parents=True, exist_ok=True)
for name, source in sources.items():
    link = DATASET_ROOT / 'training' / name
    if link.is_symlink() and link.resolve() == source.resolve():
        continue
    if link.exists() or link.is_symlink():
        raise RuntimeError(f'Refusing to overwrite {link}')
    link.symlink_to(source.resolve(), target_is_directory=True)
for split in ('train', 'val'):
    source = SPLIT_DIR / (split + '.txt')
    if not source.is_file():
        raise FileNotFoundError(source)
    destination = DATASET_ROOT / 'ImageSets' / source.name
    if destination.exists() and destination.read_bytes() != source.read_bytes():
        raise RuntimeError(f'Existing split changed: {destination}')
    shutil.copy2(source, destination)
for path in (A2_SELECTION, M54_SELECTION):
    if not path.is_file():
        raise FileNotFoundError(f'Required completed selection is missing: {path}')
    report = json.loads(path.read_text())
    checkpoint = Path(report['selected_checkpoint'])
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Checkpoint named in {path}: {checkpoint}')
print('KITTI sources:', sources)
print('Selections found. Exact checkpoint and split hashes are verified in section 4.')


## 4. Regression tests and immutable experiment manifest

Both checkpoints and splits are SHA-256 locked. Learning rate 1e-5, seed 20268, batch size 4, 10 epochs per arm, FP32 training. There is no teacher model in the training optimizer. A changed environment/source/manifest requires review or a new output directory.


In [ ]:
run_logged([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
            '-p', 'test_m61_distillation.py', '-v'], 'm61_tests', cwd=MOBILE_REPO)
run_logged([sys.executable, '-u', MOBILE_REPO / 'scripts/prepare_m61_distillation.py',
            '--student-repo', STUDENT_REPO, '--teacher-repo', TEACHER_REPO,
            '--a2-selection', A2_SELECTION, '--m54-selection', M54_SELECTION,
            '--dataset-root', DATASET_ROOT, '--output-dir', OUTPUT_ROOT],
           'prepare', cwd=MOBILE_REPO)
manifest = json.loads(MANIFEST.read_text())
print('Manifest:', MANIFEST, '\nSignature:', manifest['manifest_sha256'])
run_logged([sys.executable, '-u', MOBILE_REPO / 'scripts/probe_m61_dataset_views.py',
            '--student-repo', STUDENT_REPO, '--teacher-repo', TEACHER_REPO,
            '--dataset-root', DATASET_ROOT, '--split', 'train', '--limit', '16',
            '--output', OUTPUT_ROOT / 'm61_dataset_probe.json'], 'dataset_view_probe', cwd=MOBILE_REPO)


## 5. Cache frozen MonoDGP teacher predictions — training split only

3,712 unaugmented images, no optimizer steps. Compact outputs and input/target hashes are saved, not full image tensors. Completed sample files are reused after a restart.


In [ ]:
pilot('cache_m61_predictions.py', '--role', 'teacher')


## 6. Cache frozen A2 student predictions — the same training images

Runs in a separate process to prevent the two upstream `lib` packages from colliding.


In [ ]:
pilot('cache_m61_predictions.py', '--role', 'student')


## 7. Audit teacher quality and freeze reliable Vehicle supervision

Requires exact teacher/student image and GT hashes. Teacher Vehicle score ≥0.30, 2D IoU ≥0.50, GT depth in [2,60) m. For each geometry component, use only objects where teacher error is at least 5% lower than frozen A2. Enable a component only if its mean error over all quality-filtered paired objects is lower and it has ≥100 approved objects. No validation labels enter this selection. **If the audit fails, stop and send its JSON.**


In [ ]:
pilot('audit_m61_teacher.py')
audit = json.loads((OUTPUT_ROOT / 'm61_teacher_audit.json').read_text())
print(json.dumps({k: v for k, v in audit.items() if k != 'approved_files'}, indent=2))
assert audit['complete'] and audit['pilot_authorized']


## 8. Re-evaluate the unchanged A2 baseline on all validation images

This establishes the same-environment comparison for AP and nearby recall. It does not select teacher targets. Existing hash-verified prediction files are reused on restart; metric reports are recomputed.


In [ ]:
pilot('evaluate_m61_pilot.py', '--baseline-only')


## 9. Real CUDA forward/backward smoke — zero optimizer steps

Checks finite GT+KD loss, actual nonzero teacher-loss gradients, and approved Vehicle matches. **Training must not start if this fails.**


In [ ]:
pilot('train_m61_student.py', '--smoke')
smoke = json.loads((OUTPUT_ROOT / 'm61_training_smoke.json').read_text())
assert smoke['complete'] and smoke['finite_gradients'] and smoke['teacher_gradient_nonzero']
assert smoke['optimizer_steps'] == 0
print(json.dumps(smoke, indent=2))


## 10. Control: 10 epochs, GT-only continuation

All GT losses and both classes remain active. Progress and loss print every 20 batches; each completed epoch is saved atomically to Drive. Rerunning resumes at the last completed epoch, not from scratch.


In [ ]:
pilot('train_m61_student.py', '--variant', 'control')


## 11. Treatment: 10 epochs, GT + audited Vehicle geometry distillation

Same initial A2 checkpoint, epoch seeds, data order, unaugmented images, optimizer and learning rate. The only treatment is the additional teacher loss. Distillation weight is fixed at 0.25; no temperature or weight sweep.


In [ ]:
pilot('train_m61_student.py', '--variant', 'vehicle_kd')


## 12. Complete comparison and return bundle — stop for review

Evaluate baseline and both branches at epochs 5 and 10 on all 3,769 validation images. **Epoch 10 is the predeclared decision point**; epoch 5 is diagnostic only. Require all five existing AP gates, Vehicle 3D ≥15.8713 and ≥0.10 AP gain over control, no Pedestrian 3D/BEV/nearby-recall or balanced-mean loss versus either baseline or control, and the documented Vehicle preservation limits.

A pilot pass recommends review/confirmation only. It does not authorize longer training, full-dataset test claims, or deployment. The separate Pedestrian nearby-recall target remains 0.80. Download the result ZIP and send it back; checkpoints remain on Drive.


In [ ]:
pilot('evaluate_m61_pilot.py')
report_path = OUTPUT_ROOT / 'm61_pilot_comparison.json'
report = json.loads(report_path.read_text())
print(json.dumps(report, indent=2))
import pandas as pd
display(pd.read_csv(OUTPUT_ROOT / 'm61_pilot_comparison.csv'))
result_files = [
    MANIFEST, OUTPUT_ROOT / 'm61_teacher_audit.json', OUTPUT_ROOT / 'm61_training_smoke.json',
    OUTPUT_ROOT / 'm61_baseline_metrics.json', report_path,
    OUTPUT_ROOT / 'm61_pilot_comparison.csv',
    OUTPUT_ROOT / 'm61_control/training_summary.json',
    OUTPUT_ROOT / 'm61_vehicle_kd/training_summary.json',
]
bundle = OUTPUT_ROOT / 'm61_results.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in result_files:
        if not path.is_file():
            raise FileNotFoundError(path)
        archive.write(path, path.relative_to(OUTPUT_ROOT))
print('Return:', bundle)
print('STOP for review; no full run or phone deployment is authorized.')
from google.colab import files
files.download(str(bundle))
